# Fine-tuning com PEFT/LoRA vs. Prompt Engineering — Módulo 10 (exercício separado)

Tarefa: classificar o **tema** de uma proposição legislativa a partir da ementa (saúde, educação, segurança, meio ambiente, outro), comparando um classificador fine-tunado com LoRA contra classificação via prompt puro (zero-shot) no mesmo modelo base usado no RAG.

Rótulos construídos por palavra-chave (heurística, não é rótulo oficial da API) - suficiente para o exercício de comparar as duas técnicas, não para uso em produção.

In [1]:
import sys
sys.path.append("..")

import re
import numpy as np
import pandas as pd
import torch

from src.rag_pipeline import fetch_proposicoes_gerais

torch.manual_seed(42)

proposicoes = fetch_proposicoes_gerais(anos=[2023, 2024], max_paginas_por_ano=15)
df = pd.DataFrame(proposicoes)
df = df.dropna(subset=["ementa"])
print(f"{len(df)} proposições com ementa")

PALAVRAS_CHAVE = {
    "saude": ["saúde", "sus", "hospital", "médic", "doença", "vacina", "sanitár"],
    "educacao": ["educaç", "escola", "ensino", "universidade", "aluno", "professor"],
    "seguranca": ["segurança pública", "polícia", "crime", "violência", "armas"],
    "meio_ambiente": ["meio ambiente", "clima", "ambiental", "sustentáv", "florest"],
}

def rotular(ementa: str) -> str:
    ementa_lower = ementa.lower()
    for tema, palavras in PALAVRAS_CHAVE.items():
        if any(p in ementa_lower for p in palavras):
            return tema
    return "outro"

df["tema"] = df["ementa"].apply(rotular)
print(df["tema"].value_counts())

3000 proposições com ementa
tema
outro            2231
saude             333
seguranca         170
educacao          167
meio_ambiente      99
Name: count, dtype: int64


In [2]:
MAX_POR_CLASSE = 100
df_balanceado = (
    df.groupby("tema", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), MAX_POR_CLASSE), random_state=42), include_groups=True)
    .reset_index(drop=True)
)
print(df_balanceado["tema"].value_counts())

temas = sorted(df_balanceado["tema"].unique())
tema_para_id = {t: i for i, t in enumerate(temas)}
df_balanceado["label"] = df_balanceado["tema"].map(tema_para_id)

df_treino = df_balanceado.sample(frac=0.8, random_state=42)
df_teste = df_balanceado.drop(df_treino.index)
print(f"\nTreino: {len(df_treino)} | Teste: {len(df_teste)}")

tema
educacao         100
outro            100
saude            100
seguranca        100
meio_ambiente     99
Name: count, dtype: int64

Treino: 399 | Teste: 100


C:\Users\PDCASE\AppData\Local\Temp\ipykernel_15184\436301048.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), MAX_POR_CLASSE), random_state=42), include_groups=True)


## Fine-tuning com LoRA (BERTimbau + PEFT)

LoRA (Low-Rank Adaptation) não re-treina os pesos do modelo inteiro — congela o modelo base e adiciona pequenas matrizes treináveis "ao lado" de algumas camadas. Resultado: muito menos parâmetros pra treinar, sem precisar de GPU cara.

In [3]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import Dataset, DataLoader

MODELO_BASE = "neuralmind/bert-base-portuguese-cased"
tokenizer_bert = AutoTokenizer.from_pretrained(MODELO_BASE)
modelo_base = AutoModelForSequenceClassification.from_pretrained(MODELO_BASE, num_labels=len(temas))

config_lora = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=["query", "value"],
)
modelo_lora = get_peft_model(modelo_base, config_lora)
modelo_lora.print_trainable_parameters()


class DatasetEmentas(Dataset):
    def __init__(self, textos, labels, tokenizer):
        self.encodings = tokenizer(list(textos), truncation=True, padding=True, max_length=128, return_tensors="pt")
        self.labels = torch.tensor(list(labels))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


dataset_treino = DatasetEmentas(df_treino["ementa"], df_treino["label"], tokenizer_bert)
dataset_teste = DatasetEmentas(df_teste["ementa"], df_teste["label"], tokenizer_bert)
loader_treino_lora = DataLoader(dataset_treino, batch_size=16, shuffle=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

trainable params: 298,757 || all params: 109,225,738 || trainable%: 0.2735


In [4]:
otimizador_lora = torch.optim.AdamW(modelo_lora.parameters(), lr=3e-4)

N_EPOCAS_LORA = 4
modelo_lora.train()
for epoca in range(N_EPOCAS_LORA):
    perda_total = 0.0
    for lote in loader_treino_lora:
        otimizador_lora.zero_grad()
        saida = modelo_lora(**lote)
        saida.loss.backward()
        otimizador_lora.step()
        perda_total += saida.loss.item() * lote["labels"].size(0)
    print(f"Época {epoca+1}/{N_EPOCAS_LORA} - perda: {perda_total/len(dataset_treino):.4f}")

modelo_lora.eval()
loader_teste_lora = DataLoader(dataset_teste, batch_size=32)
acertos = 0
with torch.no_grad():
    for lote in loader_teste_lora:
        labels = lote.pop("labels")
        saida = modelo_lora(**lote)
        preditos = saida.logits.argmax(dim=1)
        acertos += (preditos == labels).sum().item()

acuracia_lora = acertos / len(dataset_teste)
print(f"\nAcurácia LoRA no teste: {acuracia_lora*100:.2f}%")

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Época 1/4 - perda: 1.5528


Época 2/4 - perda: 1.3668


Época 3/4 - perda: 0.9668


Época 4/4 - perda: 0.5808



Acurácia LoRA no teste: 86.00%


## Comparação: prompt engineering puro (zero-shot, sem treinar nada)

In [5]:
from src.rag_pipeline import _gerar_texto

def classificar_zero_shot(ementa: str) -> str:
    prompt = (
        f"Classifique o tema desta ementa de proposição legislativa em uma destas opções: "
        f"saude, educacao, seguranca, meio_ambiente, outro.\n\n"
        f"Ementa: {ementa}\n\nTema:"
    )
    saida = _gerar_texto(prompt).strip().lower()
    for tema in temas:
        if tema.replace("_", " ") in saida or tema in saida:
            return tema
    return "outro"

acertos_zero_shot = 0
for _, linha in df_teste.iterrows():
    predito = classificar_zero_shot(linha["ementa"])
    acertos_zero_shot += int(predito == linha["tema"])

acuracia_zero_shot = acertos_zero_shot / len(df_teste)
print(f"Acurácia zero-shot (prompt engineering): {acuracia_zero_shot*100:.2f}%")
print(f"Acurácia LoRA (fine-tuned): {acuracia_lora*100:.2f}%")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Acurácia zero-shot (prompt engineering): 27.00%
Acurácia LoRA (fine-tuned): 86.00%
